# CSP

# Motor-Imagery EEG Decoding with CSP, FBCSP, and EEGNet

## Project Overview

This notebook compares classical signal-processing pipelines (CSP and FBCSP) with a compact convolutional neural network (EEGNet) for classifying left- vs. right-hand motor imagery from EEG recordings of the BCI Competition IV 2a dataset through MOABB.

- **CSP + LDA:** Spatial filters EEG signals, then classifies log-variance features.
- **FBCSP + LDA:**: Learns CSP features independently across multiple frequency bands before selecting most informative features.
- **EEGNet:**: A compact CNN designed for EEG Decoding

## Dataset and evaluation design
- **Dataset:** BCI Competition IV 2a, Subject 1
- **Task:** left-hand versus right-hand motor-imagery classification
- **EEG:** 22 channels sampled at 250 Hz
- **Training data:** Session 1 (`0train`; 144 trials)
- **Test data:** Session 2 (`1test`; 144 trials)

Keeping the first and second sessions separate provides a temporally independent within-subject test set.

## Held-out session results

| Method | Test accuracy |
|---|---:|
| CSP + LDA | 84.0% |
| FBCSP + LDA | 88.2% |
| EEGNet | 87.5% |


## Takeaways

All three methods classify held-out session trials very well. In this single-subject comparision, FBCSP + LDA achieved the highest observed accuracy, whille EEGNet was a close second using minimally processed EEG data.

A natural next step is to repeat the same evaluation across all nine participants.



In [ ]:
import sys
import torch
import mne
import moabb
import braindecode
import sklearn
"""
MOABB0 -> standardized access to BCI competition dataset

"""
%pip install -q moabb mne braindecode skorch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("MNE:", mne.__version__)
print("MOABB:", moabb.__version__)
print("Braindecode:", braindecode.__version__)
print("scikit-learn:", sklearn.__version__)




Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
                     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/operations/check.py", line 42, in create_package_set_from_inst

**Import and Bandpass Filter the Data From One Individual (within-subject testing)**

In [ ]:
#X contains all trials, channels, and time samples.
#X.shape = (trials, channels, time samples)

#Y contains the labels (left or right)
#Y.shape = (trials,)
import numpy as np

from moabb.datasets import BNCI2014_001
#9 participants, 22 EEG electrodes, 2 dif days, 250 Hz sampling rate
#4 imagined movements (L, R, F, T) | participant sees cue and imagines movement for 4 seconds

from moabb.paradigms import LeftRightImagery
#selects only the left and right hand imagery trials



dataset = BNCI2014_001()
print(dataset.subject_list) # 9 participants
print(dataset.event_id) # left hand, right hand, feed, tongue
print(dataset.interval) # four second motor imagery period

#bandpass filter 8 Hz - 35 Hz (which contains the mu and beta rythyms useful for motor imagery decoding)
paradigm = LeftRightImagery(
    fmin=8,
    fmax=35,
)




"""
Each run contains 12 right and 12 left hand trials
There are 6 runs per session
There are two sessions

X.shape = (288 trials, 22 channels, ~1000 timepoints)
Y.shape = (288 labels,)

X will contain bandpass-filtered voltage time series data bc we call paradigm

"""

#Just get one subject (we can train on first day test on second day for within-subject validation
#this goes ahead and applies the bandpass filter to the data
X, y, metadata = paradigm.get_data(
    dataset=dataset,
    subjects=[1],
)

"""
The bandpass filter takes a sequence of roughly 1000 voltage measurements
and alters them to ignore components with frequencies outside of the range
"""

print("X:", X.shape)
print("y:", y.shape)
print("Classes:", np.unique(y, return_counts=True))
print(metadata.head())

[1, 2, 3, 4, 5, 6, 7, 8, 9]
{'left_hand': 1, 'right_hand': 2, 'feet': 3, 'tongue': 4}
[2, 6]
Attempting to create new mne-python configuration file:
/root/.mne/mne-python.json
Could not read the /root/.mne/mne-python.json json file during the writing. Assuming it is empty. Got: Expecting value: line 1 column 1 (char 0)
This is nemar-py 0.3.1.
Preparing to download nm000139 from https://data.nemar.org/
Retrieving 5 of 769 manifest files (16 concurrent downloads).


Overall:   0%|          | 0.00/12.6k [00:00<?, ?B/s]

Finished downloading nm000139 v1.0.2.
This is nemar-py 0.3.1.
Preparing to download nm000139 from https://data.nemar.org/
Retrieving 6 of 769 manifest files (16 concurrent downloads).


S3:   0%|          | 0.00/82.6M [00:00<?, ?B/s]

Finished downloading nm000139 v1.0.2.


/usr/local/lib/python3.13/dist-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/usr/local/lib/python3.13/dist-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/usr/local/lib/python3.13/dist-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/usr/local/lib/python3.13/dist-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/usr/local/lib/python3.13/di

X: (288, 22, 1001)
y: (288,)
Classes: (array(['left_hand', 'right_hand'], dtype='<U10'), array([144, 144]))
   subject session run
0        1  0train   0
1        1  0train   0
2        1  0train   0
3        1  0train   0
4        1  0train   0


**Split Data into Training and Testing Data**

In [ ]:
train_mask = metadata["session"] == "0train"
test_mask = metadata["session"] == "1test"

"""
0train is a marker for first session trials
1test is a marker for second session trials

a mask creates an array of booleans indicating whether or not a condition was met

Split data into training (first session) and testing (second session)

train_mask / test_mask should both have shape (144, 22, 1001)

"""

X[train_mask].shape
X[test_mask].shape


X_train = X[train_mask] #select every trial in X where train_mask was true
X_test = X[test_mask] #select every trial in X where test_mask was true
y_train = y[train_mask]
y_test = y[test_mask]


**CSP: Common Spatial Patterns**


Each trial has 22 simultaneous electrode signals at each timepoint. CSP learns linear combinations of those electrodes:


z(t) = w1x1(t) + w2x2(t) + ... + w22x22(t)

x1(t), ... , x2(t): the electrode voltages at time t

w1, ... w22: the learned weights


z(t) a new spatially filtered signal


**In terms of linear**

X is a matrix 22x1001:
22 rows (one for each electrode)
1001 columns (one for each timepoint)


One CSP spatial filter is a 1x22 row vector

zi = wXi

CSP looks for weighted combinations whose variance -- roughly, band-limited power-- is high for one class and low for the other


CSP is primarily combining different electrodes at the same timepoint, and it must learn its weights from the training data, not the test data, so that no leakage occurs.


How does CSP learn w?

CSP for each class estimates hwo the electrodes vary together using covariance matricies (Cleft, Cright), and it searches for directions w where the projected signal has very different variance between classes. seeks to maximize the ratio of

wT Cleft W / wT Cright W

normally learn several filters. for example, if we chose 4, W = 4x22 matrix, and zi would be 4x1001 matrix. if we calculate one log-variance value from each row of zi, we would have 4 numerical values for original trial


In [ ]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix


csp = CSP(
    n_components=4, #learn 4 spatial filters
    reg="oas", #stabilize covariance estimates bc relatively few trials
    log=True, #return log-variance of each CSP-filtered signal
)

"""
weights will be a 4x22 matrix, from each trial (22x1001) of training data,
we will receive 4 numerical values representing that trial (log-variance of each CSP filtered signal)
"""

#fit_transform: Learn W from the training data then create training features
#transform: reuse same W on unseen data without learning anything new
X_train_csp = csp.fit_transform(X_train, y_train)
X_test_csp = csp.transform(X_test)

"""
each should have shape 144 trials x 4 log-variance values
"""

#we are only passing our log-variance values into LDA
"""
This helps reduce overfitting. If we passed in 22x1001 voltage values per trial into LDA from
only 144 training trials, memorizing noise would be pretty easy.

CSP : 20,022 values per trial -> 4 values per trial that describe class discriminative (the ones that maximized the variance difference between classes), band-limited spacial parameters

So, LDA has way fewer parameters to estimate.

CSP can still overfit, which is why we only used 4 filters
"""
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_csp, y_train)


y_pred = lda.predict(X_test_csp)

"""
should have shape 144,
"""

correct = y_pred == y_test
accuracy = correct.mean()

print("Accuracy:", accuracy)



cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["left_hand", "right_hand"],
)

"""
true left predicted left, true left predicted right,
true right predicted left, true right predicted right

Notably, our model gets all of the true left ones correct
Our model gets only 49/72 of the true right ones correct

FBCSP could be used here to improve accuracy, as it considers different frequency bands.

Here, in ordinary CSP, we learned one set of spatial features from the combined 8-35Hz signal,
which means activity in one frequency range can obscure useful activity in another
"""

print(cm)

Computing rank from data with rank=None
    Using tolerance 37 (2.2e-16 eps * 22 dim * 7.6e+15  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=left_hand covariance using OAS
Done.
Estimating class=right_hand covariance using OAS
Done.
Accuracy: 0.8402777777777778
[[72  0]
 [23 49]]


# FBCSP

**FBCSP**

Standard CSP applies spatial filters to the data in one frequency band. This means that activity in one frequency range can obscure useful activity in another

FBCSP splits the data into different frequency bands, and each band gets its own CSP model and its own spatial weights.



In [ ]:
from moabb.paradigms import FilterBankLeftRightImagery

"""
(trials, channels, time, frequency bands)
lets say we split into 7 frequency bands
for our 288 trials, would have shape
(288, 22, 1001, 7)

if we extract 4 CSP features from each of 7 frequency hands, we will have 28 features per trial before feature selection
"""

#define these frequency bands
frequency_bands = [
    [8, 12],
    [12, 16],
    [16, 20],
    [20, 24],
    [24, 28],
    [28, 32],
    [32, 36],
]

#now we are getting the data and splitting it into freq bands
fb_paradigm = FilterBankLeftRightImagery(
    filters=frequency_bands
)


X_fb, y_fb, metadata_fb = fb_paradigm.get_data(
    dataset=dataset,
    subjects=[1],
)

print(X_fb.shape)





/usr/local/lib/python3.13/dist-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/usr/local/lib/python3.13/dist-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/usr/local/lib/python3.13/dist-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/usr/local/lib/python3.13/dist-packages/moabb/datasets/bnci/base.py:456: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = make_standard_montage("standard_1005")
/usr/local/lib/python3.13/di

(288, 22, 1001, 7)


array([[[ 0.25047783,  0.09646697, -0.07298083, ..., -0.94764381,
         -1.87030298, -2.59279192],
        [ 1.90224971,  1.44119834,  0.81487961, ..., -0.06999135,
         -0.57122859, -1.02961652],
        [ 1.6265833 ,  1.25600155,  0.74703035, ..., -1.18132089,
         -1.94989216, -2.51778238],
        ...,
        [ 3.14624829,  1.37941204, -0.51859513, ..., -1.40446907,
         -2.79043993, -3.88810087],
        [ 2.68502382,  1.19723616, -0.40623466, ..., -1.71546323,
         -2.99918205, -3.96341018],
        [ 4.20500577,  2.2613335 ,  0.08411704, ..., -0.86167711,
         -2.02757945, -2.97100625]],

       [[-4.0282253 , -3.91511417, -3.31489802, ..., -1.17380591,
         -1.56547011, -1.76220504],
        [-3.02854953, -3.38828019, -3.32800769, ..., -0.33797462,
         -1.10134735, -1.70045913],
        [-3.89304269, -4.02750175, -3.67227724, ..., -0.057488  ,
         -0.71549511, -1.25588059],
        ...,
        [-5.14031772, -5.4536042 , -5.10779918, ...,  

**Split data into testing and training**

In [ ]:
fb_train_mask = metadata_fb["session"] == "0train"
fb_test_mask = metadata_fb["session"] == "1test"

X_train_fb = X_fb[fb_train_mask] #should have shape 144x22x1001x7


#X_train_fb[:, :, :, 0] this would select all 144 training trials, all 22 channels, all 1001 timepoints, only 8-12 Hz band
X_test_fb = X_fb[fb_test_mask]


y_train_fb = y_fb[fb_train_mask]
y_test_fb = y_fb[fb_test_mask]


**Learn CSP weights / extract features for each frequency band**

In [ ]:
train_features_by_band = []
test_features_by_band = []
csp_models = []

for band_index in range((len(frequency_bands))):

  #extract only the frequency range desired
  band_train = X_train_fb[:, :, :, band_index]
  band_test = X_test_fb[:, :, :, band_index]

  #same as before, but we do it for each frequency band
  csp = CSP(
        n_components=4,
        reg="oas",
        log=True,
    )
  #learn weights
  train_features = csp.fit_transform(
        band_train,
        y_train_fb,
    )
  #use weights we learned from training
  test_features = csp.transform(
        band_test,
    )


  train_features_by_band.append(train_features) #train features is 144 trials x 4 features
  test_features_by_band.append(test_features) #same thing for test features
  csp_models.append(csp) #stores the CSP model (different one for each band)


  """
  train_features_by_band should have a list containing 7 arrays
  each array should be shaped 144x4 (144 trials, 4 log variance values for each trial in each band)

  """

#concatenate such that each trial gets 28 features
#axis 1 so (144,4) + (144, 4) + ... = (144,28)

"""
Ordinary CSP filters the EEG signal into one broad frequency range, learns spatial filters from that signal,
and passes the resulting log-variance features to a classifier

FBCSP divdes the EEG into multiple frequency bands, learns an independent CSP model within each band,
concatenates the CSP features from each band, selects the most informative features (those that maximize
the difference between classes), and passes those selected features to a classifier


"""

X_train_fbcsp = np.concatenate(
    train_features_by_band,
    axis=1,
)


X_test_fbcsp = np.concatenate(
    test_features_by_band,
    axis=1,
)


print(X_train_fbcsp.shape)
print(X_test_fbcsp.shape)


Computing rank from data with rank=None
    Using tolerance 23 (2.2e-16 eps * 22 dim * 4.8e+15  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=left_hand covariance using OAS
Done.
Estimating class=right_hand covariance using OAS
Done.
Computing rank from data with rank=None
    Using tolerance 17 (2.2e-16 eps * 22 dim * 3.4e+15  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=left_hand covariance using OAS
Done.
Estimating class=right_hand covariance using OAS
Done.
Computing rank from data with rank=None
    Using tolerance 13 (2.2e-16 eps * 22 dim * 2.7e+15  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=left_hand covariance using 

**Select the most important features to pass into the model**

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

#select the k best features to pass into classifer (of 28)
selector = SelectKBest(
    score_func=mutual_info_classif,
    k=8,
)

X_train_selected = selector.fit_transform(
    X_train_fbcsp,
    y_train_fb,
)

X_test_selected = selector.transform(
    X_test_fbcsp,
)

print(X_train_selected.shape)
print(X_test_selected.shape)

(144, 8)
(144, 8)


**Train LDA model with most important features**

In [ ]:
fbcsp_lda = LinearDiscriminantAnalysis()

fbcsp_lda.fit(
    X_train_selected,
    y_train_fb,
)

fbcsp_predictions = fbcsp_lda.predict(
    X_test_selected,
)

fbcsp_accuracy = (
    fbcsp_predictions == y_test_fb
).mean()

print("FBCSP accuracy:", fbcsp_accuracy)

print(
    confusion_matrix(
        y_test_fb,
        fbcsp_predictions,
        labels=["left_hand", "right_hand"],
    )
)

FBCSP accuracy: 0.8819444444444444
[[68  4]
 [13 59]]


In [ ]:
#determine what features (so which freq as well as which csp component)
#were considered most important


selected_indices = selector.get_support(indices=True) #get the indicies

for feature_index in selected_indices:
    band_index = feature_index // 4 #figure out frequency band based off index
    component_index = feature_index % 4 #figure out CSP component based on index
    low, high = frequency_bands[band_index]

    print(
        f"Feature {feature_index}: "
        f"{low}-{high} Hz, "
        f"CSP component {component_index + 1}"
    )


Feature 4: 12-16 Hz, CSP component 1
Feature 5: 12-16 Hz, CSP component 2
Feature 6: 12-16 Hz, CSP component 3
Feature 8: 16-20 Hz, CSP component 1
Feature 16: 24-28 Hz, CSP component 1
Feature 21: 28-32 Hz, CSP component 2
Feature 22: 28-32 Hz, CSP component 3
Feature 23: 28-32 Hz, CSP component 4




```
# This is formatted as code
```

# EEGNet

EEGNet: less likely to overfit than Deep ConvNet... more parameters means more ways to fit a signal, but also more ways to memorize noise from a small data set. Start with EEGNet first.

EEGNet should receive minimially processed data (so the 8-35 Hz) bit, and it should learn temporal and spatial filters on its own.

**Assign numerical values to classes and split into training, validation, and testing**

In [ ]:
"""
right now y_train contains words, but NN use numerical class IDs, so convert.

y_train == "right_hand" -> True for right_hand -> 1
y_train == "right_hand" -> False for left_hand -> 0

"""

y_train_nn = (y_train == "right_hand").astype(np.int64)
y_test_nn = (y_test == "right_hand").astype(np.int64)


#select run column, which contains a run identifier for each of 288 trials
run_number = metadata["run"].astype(str)


#first session first 5 runs -> training
nn_train_mask = (
    (metadata["session"] == "0train")
    & (run_number != "5")
)

#first session last run -> validation
nn_valid_mask = (
    (metadata["session"] == "0train")
    & (run_number == "5")
)

#second session -> testing
nn_test_mask = metadata["session"] == "1test"

#should be 120 train, 24 validation, 144 test
print(nn_train_mask.sum())
print(nn_valid_mask.sum())
print(nn_test_mask.sum())


"""
Use those masks to create the actual arrays
"""
X_nn_train = X[nn_train_mask].astype(np.float32) #convert 64-bit data -> 32 bit data
X_nn_valid = X[nn_valid_mask].astype(np.float32)
X_nn_test = X[nn_test_mask].astype(np.float32)

y_nn_train = (y[nn_train_mask] == "right_hand").astype(np.int64) #loss function expects 0 or 1 integer for class label
y_nn_valid = (y[nn_valid_mask] == "right_hand").astype(np.int64)
y_nn_test = (y[nn_test_mask] == "right_hand").astype(np.int64)

print(X_nn_train.shape, y_nn_train.shape)
print(X_nn_valid.shape, y_nn_valid.shape)
print(X_nn_test.shape, y_nn_test.shape)



120
24
144
(120, 22, 1001) (120,)
(24, 22, 1001) (24,)
(144, 22, 1001) (144,)


**Create the EEGNet**

In [ ]:
from braindecode.models import EEGNet

model = EEGNet(
    n_chans=22, #electrode number
    n_outputs=2, #binary classification
    n_times=1001, #timepoints
    sfreq=250, #samplign rate
    F1=8, #initially learn 8 temporal filters
    D=2,#learn two spatial filters for each temporal filter
    F2=16, #produce 16 feature maps later
    kernel_length=125, #sampling rate / 2 (each temporal filter examines .5 second section at a time)
    drop_prob=0.5, #reduce overfitting (.5 for within-subject, .25 for cross subject)
)

print(model)

Layer (type (var_name):depth-idx)                            Input Shape               Output Shape              Param #                   Kernel Shape
EEGNet (EEGNet)                                              [1, 22, 1001]             [1, 2]                    --                        --
├─Ensure4d (ensuredims): 1-1                                 [1, 22, 1001]             [1, 22, 1001, 1]          --                        --
├─Rearrange (dimshuffle): 1-2                                [1, 22, 1001, 1]          [1, 1, 22, 1001]          --                        --
├─Conv2d (conv_temporal): 1-3                                [1, 1, 22, 1001]          [1, 8, 22, 1001]          1,000                     [1, 125]
├─BatchNorm2d (bnorm_temporal): 1-4                          [1, 8, 22, 1001]          [1, 8, 22, 1001]          16                        --
├─ParametrizedConv2dWithConstraint (conv_spatial): 1-5       [1, 8, 22, 1001]          [1, 16, 1, 1001]          --                 

In [ ]:
import torch

small_batch = torch.from_numpy(X_nn_train[:5]) #converts numpy array of trials into pytorch tensor

model.eval() #switches off training only behavior

with torch.no_grad():
    output = model(small_batch)

print("Input shape:", small_batch.shape)

#should be five trials, then two columns (one score for each)
print("Output shape:", output.shape)
print(output)

Input shape: torch.Size([5, 22, 1001])
Output shape: torch.Size([5, 2])
tensor([[ 0.0258, -0.0266],
        [ 0.0198, -0.0002],
        [ 0.0441,  0.0085],
        [ 0.0234,  0.0129],
        [ 0.0104, -0.0032]])


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

#create training dataset
#convert eeg numpy array into pytorch tensor
train_dataset = TensorDataset(
    torch.from_numpy(X_nn_train),
    torch.from_numpy(y_nn_train),
)

valid_dataset = TensorDataset(
    torch.from_numpy(X_nn_valid),
    torch.from_numpy(y_nn_valid),
)


#object that will deliver training data to eegnet
train_loader = DataLoader(
    train_dataset, #which dataset to use
    batch_size=16, #how many trials processed at once
    shuffle=True, #randomly changes order of training trials at beginning of every epoch
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False,
)

batch_X, batch_y = next(iter(train_loader))



#now set up loss function and optimizer

#cuda if can use googles cuda, otherwise cpu
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
)

print(device)

print(batch_X.shape)
print(batch_y.shape)



cpu
torch.Size([16, 22, 1001])
torch.Size([16])


In [ ]:

import copy
best_valid_loss = float("inf")
best_model_state = None

for epoch in range(1, 51): #50 epochs

    # TRAINING
    model.train()

    train_loss_total = 0
    train_correct = 0
    train_count = 0

    for batch_X, batch_y in train_loader: #one batch at a time

        #moves EEG and labels to same processor as EEGNet
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        #clears gradients for previous batch (so doesn't add new ones to old ones)
        optimizer.zero_grad()


        logits = model(batch_X)
        loss = loss_function(logits, batch_y)

        #calculates a gradient for every learnable weight
        loss.backward()

        #adam uses gradients to update the weights
        optimizer.step()


        #keep track of total loss
        train_loss_total += loss.item() * batch_y.size(0)
        train_correct += (
            logits.argmax(dim=1) == batch_y
        ).sum().item()
        train_count += batch_y.size(0)

    train_loss = train_loss_total / train_count
    train_accuracy = train_correct / train_count

    # VALIDATION
    model.eval()

    valid_loss_total = 0
    valid_correct = 0
    valid_count = 0

    with torch.no_grad():
        #measures the current model without changing it
        for batch_X, batch_y in valid_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_X)
            loss = loss_function(logits, batch_y)

            valid_loss_total += loss.item() * batch_y.size(0)
            valid_correct += (
                logits.argmax(dim=1) == batch_y
            ).sum().item()
            valid_count += batch_y.size(0)

    valid_loss = valid_loss_total / valid_count
    valid_accuracy = valid_correct / valid_count

    # SAVE THE BEST VALIDATION MODEL
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_model_state = copy.deepcopy(
            model.state_dict()
        )

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.3f} | "
        f"train acc {train_accuracy:.3f} | "
        f"valid loss {valid_loss:.3f} | "
        f"valid acc {valid_accuracy:.3f}"
    )

model.load_state_dict(best_model_state)

Epoch 01 | train loss 0.679 | train acc 0.567 | valid loss 0.693 | valid acc 0.458
Epoch 02 | train loss 0.682 | train acc 0.558 | valid loss 0.692 | valid acc 0.458
Epoch 03 | train loss 0.673 | train acc 0.592 | valid loss 0.692 | valid acc 0.458
Epoch 04 | train loss 0.683 | train acc 0.542 | valid loss 0.692 | valid acc 0.458
Epoch 05 | train loss 0.670 | train acc 0.633 | valid loss 0.693 | valid acc 0.458
Epoch 06 | train loss 0.651 | train acc 0.642 | valid loss 0.693 | valid acc 0.500
Epoch 07 | train loss 0.644 | train acc 0.633 | valid loss 0.693 | valid acc 0.583
Epoch 08 | train loss 0.631 | train acc 0.692 | valid loss 0.692 | valid acc 0.500
Epoch 09 | train loss 0.642 | train acc 0.725 | valid loss 0.692 | valid acc 0.458
Epoch 10 | train loss 0.633 | train acc 0.658 | valid loss 0.693 | valid acc 0.375
Epoch 11 | train loss 0.625 | train acc 0.750 | valid loss 0.694 | valid acc 0.375
Epoch 12 | train loss 0.622 | train acc 0.658 | valid loss 0.694 | valid acc 0.417
Epoc

<All keys matched successfully>

In [ ]:
model.eval()

test_X_tensor = torch.from_numpy(X_nn_test).to(device)

with torch.no_grad():
    test_logits = model(test_X_tensor)
    test_predictions = (
        test_logits.argmax(dim=1).cpu().numpy()
    )

test_accuracy = (
    test_predictions == y_nn_test
).mean()

test_confusion = confusion_matrix(
    y_nn_test,
    test_predictions,
    labels=[0, 1],
)

print("EEGNet test accuracy:", test_accuracy)
print(test_confusion)

EEGNet test accuracy: 0.875
[[60 12]
 [ 6 66]]
